## 4D Objects-By-Change - Seed detection

This notebook explains the methods available for detecting seeds in the extraction of *4D objects-by-change* (4D-OBCs; [Anders et al., 2020](#References)) and how to choose between them. The full 4D-OBC workflow is explained in the [4D-OBC analysis notebook](4dobc-analysis.ipynb).

Implemented in: `py4dgeo.RegionGrowingAlgorithm` (parameter `method`)

**Implemented by**  
Dominic Kempf ([\@dokempf](https://github.com/dokempf), Heidelberg University): `volume_cpd`  
Robin Scharnagl (Technical University of Munich), Xiaoyu Huang ([\@xyuhuang](https://github.com/xyuhuang), Technical University of Munich): `linear_rdp`, `linear_dtr`

**Author(s) of the method**
`volume_cpd`: [Katharina Anders](https://www.professoren.tum.de/anders-katharina), [Lukas Winiwarter](https://www.uibk.ac.at/de/peak/expertinnen/lukas-winiwarter/), [Bernhard Höfle](https://www.geog.uni-heidelberg.de/en/people-at-the-institute/prof-dr-bernhard-hofle) (Heidelberg University)
`linear_rdp`, `linear_dtr`: M. Ulm, M. Elias, A. Eltner, E. Lotsari, [K. Anders](https://www.professoren.tum.de/anders-katharina)

**Original publication of the method**
`volume_cpd`: Anders, K., Winiwarter, L., Mara, H., Lindenbergh, R., Vos, S. E., & Höfle, B. (2021). Fully automatic spatiotemporal segmentation of 3D LiDAR time series for the extraction of natural surface changes. ISPRS Journal of Photogrammetry and Remote Sensing, 173, pp. 297-308. doi: [10.1016/j.isprsjprs.2021.01.015](https://doi.org/10.1016/j.isprsjprs.2021.01.015).
`linear_rdp`, `linear_dtr`: Ulm, M., Elias, M., Eltner, A., Lotsari, E., & Anders, K. (2025). Automated change detection in photogrammetric 4D point clouds – transferability and extension of 4D objects-by-change for monitoring riverbank dynamics using low-cost cameras. Applied Geomatics. doi: [10.1007/s12518-025-00623-9](https://doi.org/10.1007/s12518-025-00623-9).

A seed marks a location and a time span in which a change was detected in the time series of a core point. To detect changes of the surface, the seed detection analyzes the time series of M3C2 distances at each core point. The 4D-OBC algorithm grows its objects from these seeds, so the seed detection determines which changes are found. Three seed detection methods are available, which are selected with the `method` parameter of `RegionGrowingAlgorithm`:

- `volume_cpd` (default): detects change events with a clear start and end ([Anders et al., 2021](#References))
- `linear_rdp` and `linear_dtr`: detect sections of linear change ([Ulm et al., 2025](#References))

The methods are explained in the following sections. The following parameters are used by all of them:

- `seed_subsampling` (int): only every n-th core point is checked for seeds, which reduces the number of seed candidates and speeds up the analysis (default: `1`, i.e. all core points)
- `height_threshold` (float): minimum change required for a seed candidate to be accepted (default: `0.0`)
- `neighborhood_radius` (float): radius of the spatial neighborhood used during region growing (default: `1.0`)

In [ ]:
import py4dgeo
import os
from datetime import datetime
import pooch

First, we download the dataset and sort the point clouds by their acquisition time:

In [ ]:
# Download data from Zenodo and set path to point cloud folder
p = pooch.Pooch(base_url="doi:10.5281/zenodo.10003574/", path=pooch.os_cache("py4dgeo"))
p.load_registry_from_doi()
p.fetch("module3.zip", processor=pooch.Unzip())
pc_dir = p.path / "module3.zip.unzip" / "module3" / "schneeferner" / "pointclouds"


def parse_timestamp(fname):
    ts_str = "_".join(fname.split(".")[0].split("_")[1:])  # YYMMDD_hhmmss
    return datetime.strptime(ts_str, "%y%m%d_%H%M%S")


pc_files = sorted(
    (f for f in os.listdir(pc_dir) if f.endswith(".laz")),
    key=parse_timestamp,
)
timestamps = [parse_timestamp(f) for f in pc_files]

We then create a `SpatiotemporalAnalysis` object (see the [notebook on creating analysis files](4dobc-creation.ipynb)). The first epoch is used as reference epoch and its points as core points, and the M3C2 distances are computed for all epochs:

In [ ]:
analysis = py4dgeo.SpatiotemporalAnalysis(str(pc_dir / "schneeferner.zip"), force=True)

# Read the reference epoch and set the timestamp
reference_epoch = py4dgeo.read_from_las(str(pc_dir / pc_files[0]))
reference_epoch.timestamp = timestamps[0]

# Set the reference epoch and use its points as core points
analysis.reference_epoch = reference_epoch
analysis.corepoints = reference_epoch.cloud[::20]

# Distance calculation
analysis.m3c2 = py4dgeo.M3C2(cyl_radius=2.0, normal_radii=[2.0])
epochs_to_add = []
for fname, ts in zip(pc_files[1:], timestamps[1:]):
    epoch = py4dgeo.read_from_las(str(pc_dir / fname))
    epoch.timestamp = ts
    epochs_to_add.append(epoch)

analysis.add_epochs(*epochs_to_add)

### Change events (`volume_cpd`)

This is the original seed detection of the 4D-OBC method and the default ([Anders et al., 2021](#References)). A sliding window moves along the time series of each core point and detects change points, i.e. epochs at which the behavior of the time series changes. Starting at a change point, the change is followed until the accumulated change (the change volume) starts to decrease, which marks the end of the change event. The change event is used as seed if it lasts long enough and its magnitude exceeds `height_threshold`. Seeds whose neighboring core points show a similar change are grown first.

This method is suited for change events with a clear start and end. Its main parameters are:

- `window_width` (int): width of the sliding window in epochs (default: `24`)
- `window_penalty` (float): sensitivity of the change point detection; a higher value results in fewer change points (default: `1.0`)
- `minperiod` (int): minimum number of epochs of a change event (default: `24`)
- `use_unfinished` (bool): whether changes that are still ongoing at the last epoch are used as seeds (default: `True`)

These parameters are only used by `volume_cpd`.

As our data is acquired hourly, we only use change events that last at least two days:

In [ ]:
algo_cpd = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=2.0,
    seed_subsampling=50,
    height_threshold=0.5,
    minperiod=48,
    method="volume_cpd",
)

### Linear changes (`linear_rdp`, `linear_dtr`)

The linear seed detection methods detect linear temporal changes at each core point ([Ulm et al., 2025](#References)). Both methods aim to find and quantify sections of linear change in the time series of M3C2 distances, and they are suited for gradual changes. They differ in how the time series is divided into segments of approximately linear change (see below). For each segment, a linear regression is fitted and the segment is extended by one epoch on both sides. If the fitted change exceeds `height_threshold` and the segment is not longer than `max_change_period`, it is added to the list of seed candidates. The seed candidates are then sorted by their change magnitude, so that seeds with the largest change are grown first.

The parameters of the linear methods are:

- `max_change_period` (int): maximum number of epochs of a segment; longer segments are discarded (default: `200`)
- `data_gap` (int or None): first epoch after a known gap in the data; seeds crossing the gap are discarded (default: `None`)

#### Ramer-Douglas-Peucker (`linear_rdp`)

The Ramer-Douglas-Peucker algorithm ([Douglas & Peucker, 1973](#References)) simplifies the time series curve of each core point, dividing it into segments that represent approximately linear trends. Deviations from these segments that are smaller than the mean level of detection of the M3C2 distances are treated as noise. This method requires the Python package `rdp` (`pip install rdp`).

In [ ]:
algo_rdp = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=2.0,
    seed_subsampling=50,
    height_threshold=0.5,
    method="linear_rdp",
)

#### Decision tree regression (`linear_dtr`)

For each core point, the gradient (rate of change) of the time series is calculated. A decision tree regressor then splits the time series into regions with similar gradient values, i.e. regions with a nearly constant slope, in which the data behaves linearly. Each of these regions is used as a segment as described above. This method does not require an additional package.

In [ ]:
algo_dtr = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=2.0,
    seed_subsampling=50,
    height_threshold=0.5,
    method="linear_dtr",
)

### Running the 4D-OBC extraction

We choose one of the algorithms defined above and run it on the analysis object. This detects the seeds and grows the 4D-OBCs from them:

In [ ]:
algo = algo_rdp  # algo_cpd, algo_rdp or algo_dtr

objects = algo.run(analysis)
print(f"The segmentation extracted {len(objects)} 4D objects-by-change.")

We can then visually inspect objects like this:

In [ ]:
objects[0].plot()

The detected seeds and objects are stored in the analysis object and reused in later runs. To try another method or other parameters, remove them first:

In [ ]:
analysis.invalidate_results(seeds=True, objects=True)

The linear methods can result in several 4D-OBCs that describe the same change. These can be merged as described in the [notebook on aggregating 4D-OBCs](4dobc-aggregate.ipynb). To implement your own seed detection, see the [notebook on customizing the 4D-OBC algorithm](4dobc-customization.ipynb).

### References

* Anders, K., Winiwarter, L., Lindenbergh, R., Williams, J. G., Vos, S. E., & Höfle, B. (2020). 4D objects-by-change: Spatiotemporal segmentation of geomorphic surface change from LiDAR time series. ISPRS Journal of Photogrammetry and Remote Sensing, 159, pp. 352-363. doi: [10.1016/j.isprsjprs.2019.11.025](https://doi.org/10.1016/j.isprsjprs.2019.11.025).
* Anders, K., Winiwarter, L., Mara, H., Lindenbergh, R., Vos, S. E., & Höfle, B. (2021). Fully automatic spatiotemporal segmentation of 3D LiDAR time series for the extraction of natural surface changes. ISPRS Journal of Photogrammetry and Remote Sensing, 173, pp. 297-308. doi: [10.1016/j.isprsjprs.2021.01.015](https://doi.org/10.1016/j.isprsjprs.2021.01.015).
* Douglas, D. H., & Peucker, T. K. (1973). Algorithms for the reduction of the number of points required to represent a digitized line or its caricature. Cartographica, 10(2), pp. 112-122. doi: [10.3138/FM57-6770-U75U-7727](https://doi.org/10.3138/FM57-6770-U75U-7727).
* Ulm, M., Elias, M., Eltner, A., Lotsari, E., & Anders, K. (2025). Automated change detection in photogrammetric 4D point clouds – transferability and extension of 4D objects-by-change for monitoring riverbank dynamics using low-cost cameras. Applied Geomatics. doi: [10.1007/s12518-025-00623-9](https://doi.org/10.1007/s12518-025-00623-9).